In [2]:
# The skeleton codes for the ELEC378 Final Project to load the dataset and help you get started
# Author: Rocky Ren, Harvey Chen, Matthew Karazincir
# Gooood luck!


# Before you start, you should do an Anaconda environment setup. Search it up online, it will make collaborations 
# so much easier. 
# The libraries in your conda environment can be transferred as a .yml file, which is great.
 
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

# Modify these paths to match your dataset. 
BASE_DIR = os.getcwd()
DATA_DIR = BASE_DIR # This is the directory where you put the dataset.
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images/train_images")
CSV_PATH = os.path.join(DATA_DIR, "train.csv")

RANDOM_STATE = 42 # This is for the reproducibility of the train test split. 

# Loads the metadata from the csv file
def load_metadata():
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df)} entries, {df['TARGET'].nunique()} classes")
    return df

# Loads the images 
def load_image(filename, size):
    path = os.path.join(TRAIN_IMG_DIR, filename)
    img = Image.open(path).convert("RGB").resize((size, size))
    return np.array(img)

# This loads one image and its label of your choice. 
# Input: the dataframe, the index of the image, and the size of the image. 
def load_image_label_pair(df, index, size=224):
    row = df.iloc[index]
    img = load_image(row["file_name"], size)
    label = row["TARGET"]
    return img, label

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import pandas as pd
from torchvision.io import decode_image

df = load_metadata()

train_idxs, test_idxs = train_test_split(
    range(len(df)),
    test_size=0.2,
    stratify=df["TARGET"].values,
    random_state=RANDOM_STATE,
)

class ImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 1])
        image = Image.open(img_path).convert("RGB")
        label = self.img_labels.iloc[idx, 2]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

from torch.utils.data import DataLoader, Subset
from sklearn.preprocessing import LabelEncoder
from torchvision import transforms

# Training transforms: includes augmentation
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # Mandatory normalization for pre-trained models
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Test/Val transforms: No augmentation, just resize and normalize
test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
le = LabelEncoder()
le.fit(df["TARGET"])

def encode_target(label):       
    return int(le.transform([label])[0])

data = ImageDataset(
    annotations_file=CSV_PATH,
    img_dir=TRAIN_IMG_DIR,
    transform=train_transforms,
    target_transform=encode_target
)

train_set = Subset(data, train_idxs)
test_set = Subset(data, test_idxs)

train_dataloader = DataLoader(train_set, batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_set, batch_size=4, shuffle=False)

In [3]:
from sklearn.linear_model import SGDClassifier

metadata = load_metadata()
# load all images and labels into a single dataset for sklearn
images = []
labels = []
for idx in range(len(metadata)):
    img, label = load_image_label_pair(metadata, idx)
    images.append(img.flatten())  # Flatten the image for sklearn
    labels.append(label)

images = np.array(images)
labels = np.array(labels)

print(f"Loaded {len(images)} images and labels for sklearn")

Loaded 12594 entries, 100 classes
Loaded 12594 images and labels for sklearn


In [ ]:
from sklearn import svm
from sklearn.decomposition import PCA

# # create an incremental learning SVM classifier
# svm_clf = SGDClassifier(loss='hinge', max_iter=1000, tol=1e-3)

# svm_clf.partial_fit(images, labels, classes=np.unique(labels))

# svm_clf.score(images, labels)
pca = PCA(n_components=50)  # Reduce to 50 dimensions for visualization
reduced_images = pca.fit_transform(images)

NameError: name 'reduced_images' is not defined